# Prompt Engineering 

Before we start with the fundamentals of prompt engineering, it is essential to explore the basics of utilizing a text generation model. How do we select the model to use? Do we use a proprietary or open source model? How can we control the generated output?

Choosing a text generation model starts with choosing between proprietary models or open source models. We advise starting with a small foundation model. So let’s continue using Phi-3-mini, which has 3.8 billion parameters. This makes it suitable for running with devices up to 8 GB of VRAM. Overall, scaling up to larger models tends to be a nicer experience than scaling down.

## Loading a Text Generation Model 

In [5]:
import torch 
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Load model and tokenizer 
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", 
    device_map = device, 
    torch_dtype="auto", 
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Create a pipeline
pipe = pipeline(
    "text-generation",
    model=model, 
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500, 
    do_sample=False
)

`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
# Lets revisit the example from 1
# prompt
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."},
]

# Generate the output
output = pipe(messages)
print(output[0]["generated_text"])

You are not running the flash-attention implementation, expect numerical differences.


 Why did the chicken join the band? Because it had the drumsticks!


In [3]:
# Under the hood, transformers.pipeline first
# converts our message into a specific prompt template 

# Apply prompt template 
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


You may recognize the special tokens `<|user|>` and `<|endoftext|>` from (2). This prompt template, further illustrated below, was used during the training of the model. Not only does it provide information about who said what, but it is also used to indicate when the model should stop generating text (see the `<|end|>` token). This prompt is passed directly to the LLM and processed all at once.

![Alt-text](./figures/phi3-prompt-template.png)

## Controlling Model Output

Other than prompt engineering, we can control the kind of output we want by adjusting the model parameters (i.e `temperature` and `top_p`). These parameters control the randomness of the output. As illustrated below, in the sentence “I am driving a…” the likelihood of that sentence being followed by tokens like “car” or “truck” is generally higher than a token like “elephant.” However, there is still a possibility of “elephant” being generated but it is much lower.

![Alt-text](./figures/possibilities.png)

When we loaded our model, we purposefully set `do_sample=False` to make sure the output is somewhat consistent. This means that no sampling will be done and only the most probable next token is selected. However, to use the `temperature` and `top_p` parameters, we will set `do_sample=True` in order to make use of them.

### Temperature

The temperature controls the randomness or creativity of the text generated. It defines how likely it is to choose tokens that are less probable. The underlying idea is that a temperature of 0 generates the same response every time because it always chooses the most likely word. As illustrated below, a higher value allows less probable words to be generated.

![Alt-text](./figures/temperature.png)

As a result, a higher temperature (e.g., 0.8) generally results in a more diverse output while a lower temperature (e.g., 0.2) creates a more deterministic output.

In [4]:
# Using high temperature
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]['generated_text'])

 Why don't chickens use mobile phones? Because they don't want to peck into pecking at their contacts!


### top_p

`top_p`, also known as nucleus sampling, is a sampling technique that controls which subset of tokens (the nucleus) the LLM can consider. It will consider tokens until it reaches their cumulative probability. If we set `top_p` to 0.1, it will consider tokens until it reaches that value. If we set `top_p` to 1, it will consider all tokens. As shown below, by lowering the value, it will consider fewer tokens and generally give less “creative” output, while increasing the value allows the LLM to choose from more tokens.

![Alt-text](./figures/top_p.png)

Similarly, the `top_k` parameter controls exactly how many tokens the LLM can consider. If you change its value to 100, the LLM will only consider the top 100 most probable tokens.

In [5]:
# Using a high top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]['generated_text'])

 Why did the chicken join the comedy club? Because it wanted to be a stand-up poultry!


These parameters allow the user to have a sliding scale between being creative (high `temperature` and `top_p`) and being predictable (lower `temperature` and `top_p`).

# Intro to Prompt Engineering 

An essential part of working with text-generative LLMs is prompt engineering. By carefully designing our prompts we can guide the LLM to generate desired responses. Whether the prompts are questions, statements, or instructions, the main goal of prompt engineering is to elicit a useful response from the model. Prompt engineering is more than designing effective prompts. It can be used as a tool to evaluate the output of a model as well as to design safeguards and safety mitigation methods. This is an iterative process of prompt optimization and requires experimentation. There is not and unlikely will ever be a perfect prompt design.

We begin by answering the question: what should be in a prompt?

## The Basic Ingredients of a Prompt

An LLM is a prediction machine. Based on a certain input, the prompt, it tries to predict the words that might follow it. At its core, the prompt does not need to be more than a few words to elicit a response from the LLM.

![Alt-text](./figures/basic-prompt.png)

However, although the illustration works as a basic example, it fails to complete a specific task. Instead, we generally approach prompt engineering by asking a specific question or task the LLM should complete. To elicit the desired response, we need a more structured prompt. For example, we could ask the LLM to classify a sentence into either having positive or negative sentiment. 

![Alt-text](./figures/basic-prompt-2.png)

More complex use cases might require more components in a prompt. For instance, to make sure the model only outputs “negative” or “positive” we can introduce output indicators that help guide the model, as shown below, 

![Alt-text](./figures/basic-prompt-3.png)

We can continue adding or updating the elements of a prompt until we elicit the response we are looking for. We could add additional examples, describe the use case in more detail, provide additional context, etc. These components are merely examples and not a limited set of possibilities. The creativity that comes with designing these components is key.

## Instruction-Based Prompting

Although prompting comes in many flavors, from discussing philosophy with the LLM to role-playing with your favorite superhero, prompting is often used to have the LLM answer a specific question or resolve a certain task. This is referred to as ***instruction-based prompting***. Below illustrates a number of use cases in which instruction-based prompting plays an important role.

![Alt-text](./figures/insturction-prompting.png)

Although these tasks require different instructions, there is actually a lot of overlap in the prompting techniques used to improve the quality of the output. A non-exhaustive list of these techniques includes:

- ***Specificity***: Accurately describe what you want to achieve. Instead of asking the LLM to “Write a description for a product” ask it to “Write a description for a product in less than two sentences and use a formal tone.”
- ***Hallucination***: LLMs may generate incorrect information confidently, which is referred to as hallucination. To reduce its impact, we can ask the LLM to only generate an answer if it knows the answer. If it does not know the answer, it can respond with “I don’t know.”
- ***Order***: Either begin or end your prompt with the instruction. Especially with long prompts, information in the middle is often forgotten.1 LLMs tend to focus on information either at the beginning of a prompt (primacy effect) or the end of a prompt (recency effect).


# Advanced Prompt Engineering

On the surface, creating a good prompt might seem straightforward. Ask a specific question, be accurate, add some examples, and you are done! However, prompting can grow complex quite quickly and as a result is an often-underestimated component of leveraging LLMs. Here, we will go through several advanced techniques for building up your prompts, starting with the iterative workflow of building up complex prompts all the way to using LLMs sequentially to get improved results. Eventually, we will even build up to advanced reasoning techniques.

## The Potential Complexity of a Prompt

In our very first example, our prompt consisted of instruction, data, and output indicators. As we mentioned before, no prompt is limited to just these three components and you can build it up to be as complex as you want. Some common components are:

- ***Persona***: Describe what role the LLM should take on. For example, use “You are an expert in astrophysics” if you want to ask a question about astrophysics.
- ***Instruction***: The task itself. Make sure this is as specific as possible. We do not want to leave much room for interpretation.
- ***Context***: Additional information describing the context of the problem or task. It answers questions like “What is the reason for the instruction?”
- ***Format***: The format the LLM should use to output the generated text. Without it, the LLM will come up with a format itself, which is troublesome in automated systems.
- ***Audience***: The target of the generated text. This also describes the level of the generated output. For education purposes, it is often helpful to use ELI5 (“Explain it like I’m 5”).
- ***Tone***: The tone of voice the LLM should use in the generated text. If you are writing a formal email to your boss, you might not want to use an informal tone of voice.
- ***Data***: The main data related to the task itself.

To illustrate, let us extend the classification prompt we had earlier and use all of the preceding components. 

![Alt-text](./figures/complex-prompts.png)

This complex prompt demonstrates the modular nature of prompting. We can add and remove components freely and judge their effect on the output.

![Alt-text](./figures/complex-prompts-order.png)

The changes are not limited to simply introducing or removing components. Their order, as we saw before with the recency and primacy effects, can affect the quality of the LLM’s output. In other words, experimentation is vital when finding the best prompt for your use case. 

In [6]:
# We use ArXiv articles as our dataset. ArXiv is an open-access platform for scholarly articles
from datasets import load_dataset
dataset = load_dataset("jamescalam/llama-2-arxiv-papers")["train"]

# Extract metadata
summary = dataset["summary"]
titles = dataset["title"]
# print(titles[5] + "\n" + summary[5])

# prompt components
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
text = f"{titles[5]} \n {summary[5]}"
data = f"Text to summarize: {text}"

# The full prompt - remove and add pieces to view its impact on the generated output
query = persona + instruction + context + data_format + audience + tone + data

In [7]:
output = pipe(query, do_sample=False)
print(output[0]['generated_text'])

/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


KeyboardInterrupt: 

## In-Context Learning: Providing Examples

In the previous sections, we tried to accurately describe what the LLM should do. Instead of describing the task, why do we not just show the task? We can provide the LLM with examples of exactly the thing that we want to achieve. This is often referred to as in-context learning, where we provide the model with correct examples. Zero-shot prompting does not leverage examples, one-shot prompts use a single example, and few-shot prompts use two or more examples.

![Alt-text](./figures/in-context-prompting.png)

We can illustrate this method with a simple example taken from the original paper describing this method. The goal of the prompt is to generate a sentence with a made-up word. To improve the quality of the resulting sentence, we can show the generative model an example of what a proper sentence with a made-up word would be.


In [8]:
# Use a single example of using the made-up word in a sentence
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|end|>
<|assistant|>
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


In [9]:
# Generate the output
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

 During the medieval reenactment, the knight skillfully screeged the wooden target, impressing the onlookers with his prowess.


## Chain Prompting: Breaking up the Problem

In previous examples, we explored splitting up prompts into modular components to improve the performance of LLMs. Although this works well for many use cases, this might not be feasible for highly complex prompts or use cases. Instead of breaking the problem within a prompt, we can do so between prompts. Essentially, we take the output of one prompt and use it as input for the next, thereby creating a continuous chain of interactions that solves our problem.

To illustrate, let us say we want to use an LLM to create a product name, slogan, and sales pitch for us based on a number of product features. Although we can ask the LLM to do this in one go, we can instead break up the problem into pieces.

![Alt-text](./figures/chain-prompting.png)

This technique of chaining prompts allows the LLM to spend more time on each individual question instead of tackling the whole problem.

In [10]:
# Create name and slogan for a product
product_prompt = [
    {'role': 'user', "content": "Create a name and slogan for a chatbot that leverages LLMs."}
    
]
outputs = pipe(product_prompt)
product_description = outputs[0]['generated_text']
print(product_description)

 Name: ChatSage
Slogan: "Unleashing the power of AI to enhance your conversations."


In [11]:
# The we can use the generated output as input for the LLM
# to generate a sales pitch
sales_prompt = [
    {'role': 'user', 'content': f"Generate a very short sales pitch for the following product: '{product_description}'"}
]
outputs = pipe(sales_prompt)
sales_pitch = outputs[0]['generated_text']
print(sales_pitch)

 Introducing ChatSage, the ultimate AI companion that revolutionizes your conversations. With our cutting-edge technology, we unleash the power of AI to enhance your interactions, making every conversation more engaging and meaningful. Experience the future of communication with ChatSage today!


Although we need two calls to the model, a major benefit is that we can give each call different parameters. For instance, the number of tokens created was relatively small for the name and slogan whereas the pitch can be much longer. This can be used for a variety of use cases, including:

- ***Response validation***: Ask the LLM to double-check previously generated outputs.
- ***Parallel prompts***: Create multiple prompts in parallel and do a final pass to merge them. For example, ask multiple LLMs to generate multiple recipes in parallel and use the combined result to create a shopping list.
- ***Writing Stories***: Leverage the LLM to write books or stories by breaking down the problem into components. For example, by first writing a summary, developing characters, and building the story beats before diving into creating the dialogue.

We will automate this process and go beyond chaining LLMs. We will chain other pieces of technology together, like memory, tool use, and more! Before that, this idea of prompt chaining will be explored further in the following sections, which describe more complex prompt chaining methods like self-consistency, chain-of-thought, and tree-of-thought.

# Reasoning with Generative Models

Reasoning is a core component of human intelligence and is often compared to the emergent behavior of LLMs that often resembles reasoning. We highlight “resemble” as these models, at the time of writing, are generally considered to demonstrate this behavior through memorization of training data and pattern matching. The output that they showcase, however, can demonstrate complex behavior and although it might not be “true” reasoning, they are still referred to as reasoning capabilities.

To allow for this reasoning behavior, it is a good moment to step back and explore what reasoning entails in human behavior. To simplify, our methods of reasoning can be divided into system 1 and 2 thinking processes. System 1 thinking represents an automatic, intuitive, and near-instantaneous process. It shares similarities with generative models that automatically generate tokens without any self-reflective behavior. In contrast, system 2 thinking is a conscious, slow, and logical process, akin to brainstorming and self-reflection.

## Chain-of-Thought: Think Before Answering

Chain-of-thought aims to have the generative model “think” first rather than answering the question directly without any reasoning. As illustrated, it provides examples in a prompt that demonstrate the reasoning the model should do before generating its response. These reasoning processes are referred to as “thoughts.” This helps tremendously for tasks that involve a higher degree of complexity, like mathematical questions.

![Alt Text](./figures/chain-of-thought.png)

In [12]:
# Answering with chain of thought
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]

# generate the output
outputs = pipe(cot_prompt)
print(outputs[0]['generated_text'])

 The cafeteria started with 23 apples. They used 20 apples to make lunch, so they had 23 - 20 = 3 apples left. After buying 6 more apples, they now have 3 + 6 = 9 apples. The answer is 9.


In [13]:
# Instead of providing examples, lets ask the generative model to provide reasoning
# Common and effective mehtod is to use the phrase "Lets think step-by-step"
# Zero-shot chain-of-thought
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]

outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]['generated_text'])

 Step 1: Start with the initial number of apples in the cafeteria, which is 23.

Step 2: Subtract the number of apples used to make lunch, which is 20.
23 - 20 = 3 apples remaining.

Step 3: Add the number of apples bought, which is 6.
3 + 6 = 9 apples.

So, the cafeteria now has 9 apples.


## Self-Consistency: Sampling Outputs

Using the same prompt multiple times can lead to different results if we allow for a degree of creativity through parameters like `temperature` and `top_p`. As a result, the quality of the output might improve or degrade depending on the random selection of tokens. In other words, luck!

To counteract this degree of randomness and improve the performance of generative models, self-consistency was introduced. This method asks the generative model the same prompt multiple times and takes the majority result as the final answer. During this process, each answer can be affected by different `temperature` and `top_p` values to increase the diversity of sampling.

![Alt text](./figures/self-consistency.png)

As a result, although the method can improve performance, it becomes n times slower where n is the number of output samples.

## Tree-of-Thought: Exploring Intermediate Steps

The ideas of chain-of-thought and self-consistency only scratch the surface of what is currently being done to mimic complex reasoning. An improvement to these approaches can be found in tree-of-thought, which allows for an in-depth exploration of several ideas. The method works as follows. When faced with a problem that requires multiple reasoning steps, it often helps to break it down into pieces. At each step, and as illustrated, the generative model is prompted to explore different solutions to the problem at hand. It then votes for the best solution and continues to the next step.

![Alt text](./figures/tree-of-thought.png)

This method is tremendously helpful when needing to consider multiple paths, like when writing a story or coming up with creative ideas. A disadvantage of this method is that it requires many calls to the generative models, which slows the application significantly. Fortunately, there has been a successful attempt to convert the tree-of-thought framework into a simple prompting technique.

Instead of calling the generative model multiple times, we ask the model to mimic that behavior by emulating a conversation between multiple experts. These experts will question each other until they reach a consensus. An example of a tree-of-thought prompt is:

In [14]:
# Zero-shot tree-of-thought
zeroshot_tot_prompt = [
    {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realizes they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]

outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]['generated_text'])


 Expert 1:
Step 1: Start with the initial number of apples, which is 23.

Expert 2:
Step 1: Subtract the number of apples used for lunch, which is 20.
Step 2: Add the number of apples bought, which is 6.

Expert 3:
Step 1: Start with the initial number of apples, which is 23.
Step 2: Subtract the number of apples used for lunch, which is 20.
Step 3: Add the number of apples bought, which is 6.

Results:
All three experts arrived at the same answer:

Expert 1: 23 - 20 + 6 = 9 apples
Expert 2: (23 - 20) + 6 = 9 apples
Expert 3: (23 - 20) + 6 = 9 apples

All three experts agree that the cafeteria has 9 apples left.


# Output Verification

Systems and applications built with generative models might eventually end up in production. When that happens, it is important that we verify and control the output of the model to prevent breaking the application and to create a robust generative AI application. Reasons for validating the output might include:

- ***Structured output***: By default, most generative models create free-form text without adhering to specific structures other than those defined by natural language. Some use cases require their output to be structured in certain formats, like JSON.
- ***Valid output***: Even if we allow the model to generate structured output, it still has the capability to freely generate its content. For instance, when a model is asked to output either one of two choices, it should not come up with a third.
- ***Ethics***: Some open source generative models have no guardrails and will generate outputs that do not consider safety or ethical considerations. For instance, use cases might require the output to be free of profanity, personally identifiable information (PII), bias, cultural stereotypes, etc.
- ***Accuracy***: Many use cases require the output to adhere to certain standards or performance. The aim is to double-check whether the generated information is factually accurate, coherent, or free from hallucination.

Controlling the output of a generative model, as we explored with parameters like `top_p` and `temperature`, is not an easy feat. These models require help to generate consistent output conforming to certain guidelines. Generally, there are three ways of controlling the output of a generative model:

- ***Examples***
- ***Grammar***
- ***Fine-tuning***

Part 12 will go through the last example, ***fine-tuning***. 

## Providing Examples

A simple and straightforward method to fix the output is to provide the generative model with examples of what the output should look like.

In [15]:
# Ex: We want the generative mode lto create a character profile for an RPG game
# Zeroshot learning: providing no examples
zeroshot_prompt = [
    {"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]

outputs = pipe(zeroshot_prompt)
print(outputs[0]['generated_text'])

 ```json
{
  "name": "Aria Stormbringer",
  "class": "Warrior",
  "race": "Human",
  "level": 10,
  "attributes": {
    "strength": 18,
    "dexterity": 12,
    "constitution": 16,
    "intelligence": 8,
    "wisdom": 10,
    "charisma": 14
  },
  "skills": {
    "melee": 15,
    "ranged": 10,
    "magic": 5,
    "stealth": 12,
    "acrobatics": 10,
    "animal_handling": 8
  },
  "equipment": {
    "weapon": "Two-handed Axe",
    "armor": "Chainmail Hauberk",
    "shield": "Breastplate Shield",
    "accessories": [
      "Warrior's Talisman",
      "Leather Armor Pouch"
    ]
  },
  "background": "Aria was born into a family of warriors. She trained under the guidance of her father, a renowned knight, and has since become a skilled fighter. She is fiercely loyal to her friends and comrades, and will stop at nothing to protect them.",
  "personality": {
    "brave": 18,
    "loyal": 17,
    "honest": 15,
    "curious": 12,
    "compassionate": 10,
    "outgoing": 8
  },
  "quests": [
 

In [16]:
# Previous example is not a valid JSON and stopped generating tokens.
# Lets do an example and guide it with certain attributes
# One-shot learning: providing example
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:

{
    "description": "A SHORT DESCRIPTION",
    "name": "THE CHARACTER'S NAME",
    "armor": "ONE PIECE OF ARMOR",
    "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]

outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

 {
    "description": "A cunning rogue with a mysterious past, skilled in stealth and deception.",
    "name": "Shadowcloak",
    "armor": "Leather Hood",
    "weapon": "Dagger"
}


## Grammar: Constrained Sampling

Few-shot learning has a big disadvantage: we cannot explicitly prevent certain output from being generated. Although we guide the model and give it instructions, it might still not follow it entirely. Instead, packages have been rapidly developed to constrain and validate the output of generative models, like Guidance, Guardrails, and LMQL. In part, they leverage generative models to validate their own output, as illustrated. The generative models retrieve the output as new prompts and attempt to validate it based on a number of predefined guardrails.

![Alt text](./figures/grammar-prompt.png)

This process can be taken one step further and instead of validating the output we can already perform validation during the token sampling process. When sampling tokens, we can define a number of grammars or rules that the LLM should adhere to when choosing its next token. For instance, if we ask the model to either return “positive,” “negative,” or “neutral” when performing sentiment classification, it might still return something else. As illustrated, by constraining the sampling process, we can have the LLM only output what we are interested in. Note that this is still affected by parameters such as `top_p` and `temperature`.

![Alt text](./figures/constrained-sampling-prompt.png)

Let us illustrate this phenomenon with llama-cpp-python, a library similar to transformers that we can use to load in our language model. It is generally used to efficiently load and use compressed models (through quantization; see part 12) but we can also use it to apply a JSON grammar. We load the same model we used throughout this chapter but use a different format instead, namely GGUF. llama-cpp-python expects this format, which is generally used for compressed (quantized) models.

In [6]:
import gc
import torch
del model, tokenizer, pipe

# Flush memory
gc.collect()
torch.mps.empty_cache()

In [7]:
# Now that we have cleared the memory, we can load Phi-3
# Load Phi-3
from llama_cpp.llama import Llama

llm = Llama.from_pretrained(
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="*fp16.gguf", 
    n_gpu_layers=-1, # indicate that we want all layers of the model
    n_ctx=2048,
    verbose=False
)

Phi-3-mini-4k-instruct-fp16.gguf:   0%|          | 0.00/7.64G [00:00<?, ?B/s]

In [8]:
# we only need to specify response_format as a JSON object 
# Generate output
output = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Create a warrior for an RPG in JSON format."},
    ],
    response_format={"type": "json_object"},
    temperature=0,
)['choices'][0]['message']["content"]

In [9]:
# To check the output is JSON
# we can process it
import json 

json_output= json.dumps(json.loads(output), indent=4)
print(json_output)

{
    "warrior": {
        "name": "Eldric Stormbringer",
        "class": "Warrior",
        "level": 5,
        "attributes": {
            "strength": 18,
            "dexterity": 9,
            "constitution": 20,
            "intelligence": 7,
            "wisdom": 6,
            "charisma": 10
        },
        "skills": [
            {
                "name": "Martial Arts",
                "proficiency": 95
            },
            {
                "name": "Swordsmanship",
                "proficiency": 98
            },
            {
                "name": "Shield Blocking",
                "proficiency": 92
            },
            {
                "name": "Heavy Armor Proficiency",
                "proficiency": 90
            }
        ],
        "equipment": [
            {
                "type": "Weapon",
                "name": "Iron Longsword",
                "damage": 12,
                "durability": 85
            },
            {
                "type": "A